# 03 — Frequency and TF-IDF Analysis

This clean notebook performs the core lexical comparison:

1. Filter the word-level table to content words.
2. Count lemma frequencies by genre.
3. Use TF-IDF to identify distinctive lexemes for each genre.
4. Export tables and figures for the paper/poster.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ALPcourse_Biblical_Hebrew_Project/biblical_hebrew_genre_analysis")
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_DIR / "output"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
DOCS_DIR = PROJECT_DIR / "docs"
POSTER_DIR = PROJECT_DIR / "poster"

for folder in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, DOCS_DIR, POSTER_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.chdir(NOTEBOOK_DIR)
print("Project directory:", PROJECT_DIR)
print("Current folder:", os.getcwd())


!pip -q install pandas scikit-learn matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

## 2. Load the processed token table

In [ ]:
tokens_path = PROCESSED_DIR / "biblical_hebrew_tokens.csv"
tokens = pd.read_csv(tokens_path)

print("Total tokens:", len(tokens))
print("Columns:", list(tokens.columns))
print("\nTokens by genre:")
print(tokens.groupby("genre").size().sort_values(ascending=False))

tokens.head()

## 3. Filter to content words

The analysis focuses on nouns, verbs, adjectives, and adverbs. It excludes function-word categories such as particles, prepositions, conjunctions, articles, pronouns, and proper names.

This makes the genre comparison more interpretable because it emphasizes lexical content rather than grammatical structure.

In [ ]:
CONTENT_POS = {"subs", "verb", "adjv", "advb"}

content = tokens[tokens["pos"].isin(CONTENT_POS)].copy()
content = content.dropna(subset=["lex"])

content["lex"] = content["lex"].astype(str).str.strip()
content["lex_clean"] = content["lex"]

content.to_csv(PROCESSED_DIR / "biblical_hebrew_content_tokens.csv", index=False)

print("Content tokens:", len(content))
print("\nContent tokens by genre:")
print(content.groupby("genre").size().sort_values(ascending=False))

content.head()

## 4. Lemma frequencies by genre

In [ ]:
genre_word_frequencies = (
    content
    .groupby(["genre", "lex_clean"])
    .size()
    .reset_index(name="frequency")
    .sort_values(["genre", "frequency"], ascending=[True, False])
)

genre_word_frequencies.to_csv(TABLES_DIR / "genre_word_frequencies.csv", index=False)

top_frequencies = genre_word_frequencies.groupby("genre").head(20)
top_frequencies.head(20)

## 5. TF-IDF by genre

In [ ]:
# Build readable labels for figures and tables.
# The computational analysis still uses BHSA lexeme codes; glosses are used only for display.

def clean_lexeme(lex):
    return str(lex).strip()

# Use the BHSA/Text-Fabric gloss column from the processed token table.
gloss_lookup = (
    content.dropna(subset=["lex_clean", "gloss"])
    .assign(lex_clean=lambda df: df["lex_clean"].astype(str).str.strip(),
            gloss=lambda df: df["gloss"].astype(str).str.strip())
    .drop_duplicates("lex_clean")
    .set_index("lex_clean")["gloss"]
    .to_dict()
)

# Manual fallbacks / clearer labels for a few common or difficult lexemes.
MANUAL_GLOSSES = {
    "MNXH/": "offering",
    "KOHN/": "priest",
    "QRBN/": "offering",
    "QDC=/": "holiness",
    "MXY/": "command",
    ">C/": "fire",
    "MW<D/": "appointed time",
    "FR<H/": "Pharaoh",
    "XLM/": "dream",
    "YRD/": "go down",
    ">DWN/": "lord/master",
    "MLK/": "king",
    "RC</": "wicked",
    "LB/": "heart",
    "YCR/": "righteous",
    "XKM/": "wise",
    "MZMWR/": "psalm",
    "DRK/": "way",
    "MWSR/": "instruction",
    "NPC/": "soul/life",
    "N>M/": "oracle",
    "YHWH/": "YHWH",
    "CB>/": "army/host",
    "KH/": "thus",
    ">RMWN/": "palace",
    "GLH/": "exile",
    "XRB/": "sword",
    "RWX/": "spirit",
}

def display_label(lex):
    lex = clean_lexeme(lex)
    gloss = MANUAL_GLOSSES.get(lex) or gloss_lookup.get(lex) or lex
    return f"{gloss} ({lex})"


genre_documents = (
    content
    .groupby("genre")["lex_clean"]
    .apply(lambda values: " ".join(values.astype(str)))
    .sort_index()
)

vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    token_pattern=None,
    min_df=1,
)

tfidf_matrix = vectorizer.fit_transform(genre_documents.values)
terms = vectorizer.get_feature_names_out()
genres = genre_documents.index.tolist()

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), index=genres, columns=terms)

tfidf_long = (
    tfidf_df
    .reset_index(names="genre")
    .melt(id_vars="genre", var_name="lex_clean", value_name="tfidf")
    .sort_values(["genre", "tfidf"], ascending=[True, False])
)

tfidf_long["display"] = tfidf_long["lex_clean"].map(display_label)

tfidf_long.to_csv(TABLES_DIR / "tfidf_by_genre.csv", index=False)

top_15_tfidf = tfidf_long.groupby("genre").head(15).reset_index(drop=True)
top_15_tfidf.to_csv(TABLES_DIR / "top_15_tfidf_by_genre.csv", index=False)

top_15_tfidf.head(30)


## 6. Summary table with interpretation

In [ ]:
INTERPRETATIONS = {
    "Law": "Ritual, priestly, sacrificial, cultic, and legal vocabulary.",
    "Narrative": "Characters, movement, dreams, speech situations, and political authority.",
    "Poetry_Wisdom": "Moral, psychological, reflective, and didactic vocabulary.",
    "Prophecy": "Divine speech, political language, military imagery, and judgment.",
}

summary_rows = []
for genre, group in top_15_tfidf.groupby("genre"):
    summary_rows.append({
        "genre": genre,
        "top_lemmas": ", ".join(group["lex_clean"].head(12)),
        "readable_labels": ", ".join(group["display"].head(12)),
        "interpretation": INTERPRETATIONS.get(genre, ""),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(TABLES_DIR / "top_tfidf_summary_by_genre.csv", index=False)
summary.to_csv(TABLES_DIR / "top_tfidf_summary_by_genre_with_interpretation.csv", index=False)

summary

## 7. Figures

In [ ]:
# Remove old, less readable figure versions from earlier drafts.
for old in FIGURES_DIR.glob("top_tfidf_*_clean.png"):
    old.unlink()

for genre, group in top_15_tfidf.groupby("genre"):
    plot_data = group.sort_values("tfidf", ascending=True).tail(15)

    plt.figure(figsize=(10, 7))
    plt.barh(plot_data["display"], plot_data["tfidf"])
    plt.xlabel("TF-IDF score")
    plt.ylabel("Lexeme gloss and BHSA code")
    plt.title(f"Top distinctive lexemes in {genre} — TF-IDF")
    plt.tight_layout()

    filename = FIGURES_DIR / f"top_tfidf_{genre}_friendly.png"
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", filename)


## 8. Interpretation checkpoint

The results should be interpreted as exploratory. TF-IDF identifies terms that are distinctive within this selected pilot sample, not universally distinctive across all Biblical Hebrew.